In [71]:
import pandas as pd
import numpy as np

test_df = pd.read_csv('data/original_data/ais_test.csv')
train_df = pd.read_csv('data/original_data/ais_train.csv', sep='|')


train_df


train_df.head()


,time,cog,sog,rot,heading,navstat,etaRaw,latitude,longitude,vesselId,portId
0,2024-01-01 00:00:25,284.0,0.7,0,88,0,01-09 23:00,-34.74370,-57.85130,61e9f3a8b937134a3c4bfdf7,61d371c43aeaecc07011a37f
1,2024-01-01 00:00:36,109.6,0.0,-6,347,1,12-29 20:00,8.89440,-79.47939,61e9f3d4b937134a3c4bff1f,634c4de270937fc01c3a7689
2,2024-01-01 00:01:45,111.0,11.0,0,112,0,01-02 09:00,39.19065,-76.47567,61e9f436b937134a3c4c0131,61d3847bb7b7526e1adf3d19
3,2024-01-01 00:03:11,96.4,0.0,0,142,1,12-31 20:00,-34.41189,151.02067,61e9f3b4b937134a3c4bfe77,61d36f770a1807568ff9a126
4,2024-01-01 00:03:51,214.0,19.7,0,215,0,01-25 12:00,35.88379,-5.91636,61e9f41bb937134a3c4c0087,634c4de270937fc01c3a74f3


In [ ]:
import pandas as pd

train_df = train_df.sort_values(by=['vesselId', 'time']).reset_index(drop=True)

train_df['last_known_latitude'] = train_df.groupby('vesselId')['latitude'].shift(1)
train_df['last_known_longitude'] = train_df.groupby('vesselId')['longitude'].shift(1)
train_df['time_of_last_known_position'] = train_df.groupby('vesselId')['time'].shift(1)

train_df = train_df.dropna()
train_df = train_df.reset_index(drop=True)

train_df.head(20)

,time,cog,sog,rot,heading,navstat,etaRaw,latitude,longitude,vesselId,portId,last_known_latitude,last_known_longitude,time_of_last_known_position
0,2024-01-12 14:31:00,307.6,17.3,5,313,0,01-14 23:30,7.57302,77.49505,61e9f38eb937134a3c4bfd8b,61d376d893c6feb83e5eb546,7.50361,77.58340,2024-01-12 14:07:47
1,2024-01-12 14:57:23,306.8,16.9,5,312,0,01-14 23:30,7.65043,77.39404,61e9f38eb937134a3c4bfd8b,61d376d893c6feb83e5eb546,7.57302,77.49505,2024-01-12 14:31:00
2,2024-01-12 15:18:48,307.9,16.9,6,313,0,01-14 23:30,7.71275,77.31394,61e9f38eb937134a3c4bfd8b,61d376d893c6feb83e5eb546,7.65043,77.39404,2024-01-12 14:57:23
3,2024-01-12 15:39:47,307.0,16.3,7,313,0,01-14 23:30,7.77191,77.23585,61e9f38eb937134a3c4bfd8b,61d376d893c6feb83e5eb546,7.71275,77.31394,2024-01-12 15:18:48
4,2024-01-12 15:54:48,307.6,16.1,5,313,0,01-14 23:30,7.81285,77.18147,61e9f38eb937134a3c4bfd8b,61d376d893c6feb83e5eb546,7.77191,77.23585,2024-01-12 15:39:47
5,2024-01-12 16:14:59,309.5,16.1,-6,313,0,01-14 23:30,7.86929,77.11032,61e9f38eb937134a3c4bfd8b,61d376d893c6feb83e5eb546,7.81285,77.18147,2024-01-12 15:54:48
6,2024-01-12 16:35:24,308.7,16.0,2,311,0,01-14 23:30,7.92585,77.03811,61e9f38eb937134a3c4bfd8b,61d376d893c6feb83e5eb546,7.86929,77.11032,2024-01-12 16:14:59
7,2024-01-12 16:55:24,310.4,16.0,-1,311,0,01-14 23:30,7.98258,76.96880,61e9f38eb937134a3c4bfd8b,61d376d893c6feb83e5eb546,7.92585,77.03811,2024-01-12 16:35:24
8,2024-01-12 17:14:36,307.5,16.1,6,307,0,01-14 23:30,8.03598,76.90095,61e9f38eb937134a3c4bfd8b,61d376d893c6feb83e5eb546,7.98258,76.96880,2024-01-12 16:55:24
9,2024-01-12 17:36:36,322.2,16.2,-4,319,0,01-14 23:30,8.10476,76.83078,61e9f38eb937134a3c4bfd8b,61d376d893c6feb83e5eb546,8.03598,76.90095,2024-01-12 17:14:36


In [73]:
import pandas as pd

# Reference times
reference_start_time = pd.to_datetime("2024-01-01 00:00:00")
reference_end_time = pd.to_datetime("2025-01-01 00:00:00")  # Might need to adjust the end time
total_time_span = (reference_end_time - reference_start_time).total_seconds()

# Training data - time feature conversion and normalization
train_df['original_time_converted'] = pd.to_datetime(train_df['time'], errors='coerce')
train_df['time_of_last_known_position_converted'] = pd.to_datetime(train_df['time_of_last_known_position'], errors='coerce')
train_df['time'] = (train_df['original_time_converted'] - reference_start_time).dt.total_seconds()
train_df['time_of_last_known_position'] = (train_df['time_of_last_known_position_converted'] - reference_start_time).dt.total_seconds()

# Normalize time (already between 0 and 1)
train_df['time'] = train_df['time'] / total_time_span
train_df['time_of_last_known_position'] = train_df['time_of_last_known_position'] / total_time_span

# Add new features
train_df['month_of_the_year'] = train_df['original_time_converted'].dt.month  # Month (1-12)
train_df['week_of_the_year'] = train_df['original_time_converted'].dt.isocalendar().week  # Week (1-53)
train_df['day_of_the_year'] = train_df['original_time_converted'].dt.dayofyear  # Day of year (1-365)
train_df['day_of_the_month'] = train_df['original_time_converted'].dt.day  # Day of month (1-31)
train_df['day_of_the_week'] = train_df['original_time_converted'].dt.dayofweek  # Day of week (0-6, where 0 is Monday)
train_df['hour_of_the_day'] = train_df['original_time_converted'].dt.hour  # Hour (0-23)
train_df['hours_passed'] = train_df['original_time_converted'].diff().dt.total_seconds() / 3600
hours = train_df['original_time_converted'].dt.hour
minutes = train_df['original_time_converted'].dt.minute

# Convert hour to cyclical features
train_df['hour_sin'] = (np.sin(2 * np.pi * hours / 24) + 1) / 2
train_df['hour_cos'] = (np.cos(2 * np.pi * hours / 24) + 1) / 2

# Convert minute to cyclical features and normalize to [0,1]
train_df['minute_sin'] = (np.sin(2 * np.pi * minutes / 60) + 1) / 2
train_df['minute_cos'] = (np.cos(2 * np.pi * minutes / 60) + 1) / 2

# Normalize other features
# Min-max normalization to scale features between 0 and 1
train_df['month_of_the_year'] = (train_df['month_of_the_year'] - 1) / 11  # Normalize month (1-12)
train_df['week_of_the_year'] = (train_df['week_of_the_year'] - 1) / 52  # Normalize week (1-53)
train_df['day_of_the_year'] = (train_df['day_of_the_year'] - 1) / 365  # Normalize day of year (1-365)
train_df['day_of_the_month'] = (train_df['day_of_the_month'] - 1) / 30  # Normalize day of month (1-31)
train_df['day_of_the_week'] = train_df['day_of_the_week'] / 6  # Normalize day of week (0-6)
train_df['hour_of_the_day'] = train_df['hour_of_the_day'] / 23  # Normalize hour (0-23)
train_df['hours_passed'] = train_df['hours_passed'] / 24

# Add time diff features
train_df = train_df.sort_values(['vesselId', 'original_time_converted'])
train_df['time_diff'] = train_df.groupby('vesselId')['original_time_converted'].diff(-1)  # Using -1 for difference with next row

# Create boolean features for different time thresholds
train_df['time_diff_gt_10min'] = ((-train_df['time_diff'].dt.total_seconds()) > 10 * 60).astype(int)
train_df['time_diff_gt_20min'] = ((-train_df['time_diff'].dt.total_seconds()) > 20 * 60).astype(int)
train_df['time_diff_gt_40min'] = ((-train_df['time_diff'].dt.total_seconds()) > 40 * 60).astype(int)
train_df['time_diff_gt_1hour'] = ((-train_df['time_diff'].dt.total_seconds()) > 1 * 3600).astype(int)
train_df['time_diff_gt_2hours'] = ((-train_df['time_diff'].dt.total_seconds()) > 2 * 3600).astype(int)
train_df['time_diff_gt_6hours'] = ((-train_df['time_diff'].dt.total_seconds()) > 6 * 3600).astype(int)
train_df['time_diff_gt_12hours'] = ((-train_df['time_diff'].dt.total_seconds()) > 12 * 3600).astype(int)
train_df['time_diff_gt_1day'] = ((-train_df['time_diff'].dt.total_seconds()) > 24 * 3600).astype(int)

# Fill NA values (last row of each vessel group) with 0
time_diff_columns = [col for col in train_df.columns if col.startswith('time_diff_gt_')]
train_df[time_diff_columns] = train_df[time_diff_columns].fillna(0)

# Drop the temporary time_diff column
train_df.drop('time_diff', axis=1, inplace=True)


# Drop intermediate columns
train_df.drop(['original_time_converted', 'time_of_last_known_position_converted'], axis=1, inplace=True)

# Test data - time feature conversion and normalization
test_df['time_converted'] = pd.to_datetime(test_df['time'], errors='coerce')
test_df['time'] = (test_df['time_converted'] - reference_start_time).dt.total_seconds()
test_df['time'] = test_df['time'] / total_time_span

# Add new features
test_df['month_of_the_year'] = test_df['time_converted'].dt.month  # Month (1-12)
test_df['week_of_the_year'] = test_df['time_converted'].dt.isocalendar().week  # Week (1-53)
test_df['day_of_the_year'] = test_df['time_converted'].dt.dayofyear  # Day of year (1-365)
test_df['day_of_the_month'] = test_df['time_converted'].dt.day  # Day of month (1-31)
test_df['day_of_the_week'] = test_df['time_converted'].dt.dayofweek  # Day of week (0-6, where 0 is Monday)
test_df['hour_of_the_day'] = test_df['time_converted'].dt.hour  # Hour (0-23)

test_df['hours_passed'] = test_df['time_converted'].diff().dt.total_seconds() / 3600
hours = test_df['time_converted'].dt.hour
minutes = test_df['time_converted'].dt.minute

# Convert hour to cyclical features
test_df['hour_sin'] = (np.sin(2 * np.pi * hours / 24) + 1) / 2
test_df['hour_cos'] = (np.cos(2 * np.pi * hours / 24) + 1) / 2

# Convert minute to cyclical features and normalize to [0,1]
test_df['minute_sin'] = (np.sin(2 * np.pi * minutes / 60) + 1) / 2
test_df['minute_cos'] = (np.cos(2 * np.pi * minutes / 60) + 1) / 2


# Normalize other features in test data
test_df['month_of_the_year'] = (test_df['month_of_the_year'] - 1) / 11  # Normalize month (1-12)
test_df['week_of_the_year'] = (test_df['week_of_the_year'] - 1) / 52  # Normalize week (1-53)
test_df['day_of_the_year'] = (test_df['day_of_the_year'] - 1) / 365  # Normalize day of year (1-365)
test_df['day_of_the_month'] = (test_df['day_of_the_month'] - 1) / 30  # Normalize day of month (1-31)
test_df['day_of_the_week'] = test_df['day_of_the_week'] / 6  # Normalize day of week (0-6)
test_df['hour_of_the_day'] = test_df['hour_of_the_day'] / 23  # Normalize hour (0-23)

# Add time diff features
test_df = test_df.sort_values(['vesselId', 'time_converted'])
test_df['time_diff'] = test_df.groupby('vesselId')['time_converted'].diff(-1)  # Using -1 for difference with next row


# Create boolean features for different time thresholds
test_df['time_diff_gt_10min'] = ((-test_df['time_diff'].dt.total_seconds()) > 10 * 60).astype(int)
test_df['time_diff_gt_20min'] = ((-test_df['time_diff'].dt.total_seconds()) > 20 * 60).astype(int)
test_df['time_diff_gt_40min'] = ((-test_df['time_diff'].dt.total_seconds()) > 40 * 60).astype(int)
test_df['time_diff_gt_1hour'] = ((-test_df['time_diff'].dt.total_seconds()) > 1 * 3600).astype(int)
test_df['time_diff_gt_2hours'] = ((-test_df['time_diff'].dt.total_seconds()) > 2 * 3600).astype(int)
test_df['time_diff_gt_6hours'] = ((-test_df['time_diff'].dt.total_seconds()) > 6 * 3600).astype(int)
test_df['time_diff_gt_12hours'] = ((-test_df['time_diff'].dt.total_seconds()) > 12 * 3600).astype(int)
test_df['time_diff_gt_1day'] = ((-test_df['time_diff'].dt.total_seconds()) > 24 * 3600).astype(int)

# Fill NA values (last row of each vessel group) with 0
time_diff_columns = [col for col in test_df.columns if col.startswith('time_diff_gt_')]
test_df[time_diff_columns] = test_df[time_diff_columns].fillna(0)

# Drop the temporary time_diff column
test_df.drop('time_diff', axis=1, inplace=True)

# Drop intermediate columns
test_df.drop(['time_converted'], axis=1, inplace=True)


In [74]:
train_df.tail()

,time,cog,sog,rot,heading,navstat,etaRaw,latitude,longitude,vesselId,...,minute_sin,minute_cos,time_diff_gt_10min,time_diff_gt_20min,time_diff_gt_40min,time_diff_gt_1hour,time_diff_gt_2hours,time_diff_gt_6hours,time_diff_gt_12hours,time_diff_gt_1day
1519758,0.349568,324.1,13.5,-2,325,0,05-08 03:00,59.63337,21.43237,clh6aqawa0007gh0z9h6zi9bo,...,0.206107,0.095492,1,1,0,0,0,0,0,0
1519759,0.349607,324.2,13.3,-3,326,0,05-08 03:00,59.69588,21.34225,clh6aqawa0007gh0z9h6zi9bo,...,0.345492,0.975528,1,1,0,0,0,0,0,0
1519760,0.349647,356.5,12.2,-1,354,0,05-08 03:00,59.76388,21.35317,clh6aqawa0007gh0z9h6zi9bo,...,0.989074,0.396044,1,1,0,0,0,0,0,0
1519761,0.349685,52.6,17.3,3,50,0,05-08 03:00,59.83316,21.38489,clh6aqawa0007gh0z9h6zi9bo,...,0.128428,0.165435,1,1,0,0,0,0,0,0
1519762,0.349725,53.6,17.7,-1,51,0,05-08 03:00,59.89167,21.54685,clh6aqawa0007gh0z9h6zi9bo,...,0.447736,0.997261,0,0,0,0,0,0,0,0


In [75]:
test_df.tail()

,ID,vesselId,time,scaling_factor,month_of_the_year,week_of_the_year,day_of_the_year,day_of_the_month,day_of_the_week,hour_of_the_day,...,minute_sin,minute_cos,time_diff_gt_10min,time_diff_gt_20min,time_diff_gt_40min,time_diff_gt_1hour,time_diff_gt_2hours,time_diff_gt_6hours,time_diff_gt_12hours,time_diff_gt_1day
51161,51161,clh6aqawa0007gh0z9h6zi9bo,0.363232,0.1,0.363636,0.346154,0.361644,0.366667,1.0,0.956522,...,0.165435,0.128428,1,1,0,0,0,0,0,0
51302,51302,clh6aqawa0007gh0z9h6zi9bo,0.363270,0.1,0.363636,0.346154,0.361644,0.366667,1.0,0.956522,...,0.396044,0.989074,1,1,0,0,0,0,0,0
51444,51444,clh6aqawa0007gh0z9h6zi9bo,0.363309,0.1,0.363636,0.346154,0.361644,0.366667,1.0,1.000000,...,0.975528,0.345492,1,1,0,0,0,0,0,0
51595,51595,clh6aqawa0007gh0z9h6zi9bo,0.363349,0.1,0.363636,0.346154,0.361644,0.366667,1.0,1.000000,...,0.095492,0.206107,1,0,0,0,0,0,0,0
51654,51654,clh6aqawa0007gh0z9h6zi9bo,0.363386,0.1,0.363636,0.346154,0.361644,0.366667,1.0,1.000000,...,0.447736,0.997261,0,0,0,0,0,0,0,0


In [76]:
# Interpolation of default cog values and normalization

train_df['cog'] = train_df['cog'].replace(360, pd.NA)

train_df['cog'] = pd.to_numeric(train_df['cog'], errors='coerce')

train_df['cog'] = train_df['cog'].interpolate(method='linear')

train_df['cog'] = train_df['cog'].ffill()
train_df['cog'] = train_df['cog'].bfill()

print("Max COG value:", train_df['cog'].max())
print("Min COG value:", train_df['cog'].min())
print("Number of NaN values:", train_df['cog'].isna().sum())

train_df['cog'] = train_df['cog'] / 359.9

train_df.tail()

Max COG value: 359.9
Min COG value: 0.0
Number of NaN values: 0


,time,cog,sog,rot,heading,navstat,etaRaw,latitude,longitude,vesselId,...,minute_sin,minute_cos,time_diff_gt_10min,time_diff_gt_20min,time_diff_gt_40min,time_diff_gt_1hour,time_diff_gt_2hours,time_diff_gt_6hours,time_diff_gt_12hours,time_diff_gt_1day
1519758,0.349568,0.900528,13.5,-2,325,0,05-08 03:00,59.63337,21.43237,clh6aqawa0007gh0z9h6zi9bo,...,0.206107,0.095492,1,1,0,0,0,0,0,0
1519759,0.349607,0.900806,13.3,-3,326,0,05-08 03:00,59.69588,21.34225,clh6aqawa0007gh0z9h6zi9bo,...,0.345492,0.975528,1,1,0,0,0,0,0,0
1519760,0.349647,0.990553,12.2,-1,354,0,05-08 03:00,59.76388,21.35317,clh6aqawa0007gh0z9h6zi9bo,...,0.989074,0.396044,1,1,0,0,0,0,0,0
1519761,0.349685,0.146152,17.3,3,50,0,05-08 03:00,59.83316,21.38489,clh6aqawa0007gh0z9h6zi9bo,...,0.128428,0.165435,1,1,0,0,0,0,0,0
1519762,0.349725,0.148930,17.7,-1,51,0,05-08 03:00,59.89167,21.54685,clh6aqawa0007gh0z9h6zi9bo,...,0.447736,0.997261,0,0,0,0,0,0,0,0


In [77]:
# Interpolation of default sog values and normalization

train_df['sog'] = train_df['sog'].replace(102.3, pd.NA)

train_df['sog'] = pd.to_numeric(train_df['sog'], errors='coerce')

train_df['sog'] = train_df['sog'].interpolate(method='linear')

train_df['sog'] = train_df['sog'].ffill()
train_df['sog'] = train_df['sog'].bfill()

print("Max SOG value:", train_df['sog'].max())
print("Min SOG value:", train_df['sog'].min())
print("Number of NaN values:", train_df['sog'].isna().sum())

train_df['sog'] = train_df['sog'] / 102.2

train_df.tail()

Max SOG value: 102.2
Min SOG value: 0.0
Number of NaN values: 0


,time,cog,sog,rot,heading,navstat,etaRaw,latitude,longitude,vesselId,...,minute_sin,minute_cos,time_diff_gt_10min,time_diff_gt_20min,time_diff_gt_40min,time_diff_gt_1hour,time_diff_gt_2hours,time_diff_gt_6hours,time_diff_gt_12hours,time_diff_gt_1day
1519758,0.349568,0.900528,0.132094,-2,325,0,05-08 03:00,59.63337,21.43237,clh6aqawa0007gh0z9h6zi9bo,...,0.206107,0.095492,1,1,0,0,0,0,0,0
1519759,0.349607,0.900806,0.130137,-3,326,0,05-08 03:00,59.69588,21.34225,clh6aqawa0007gh0z9h6zi9bo,...,0.345492,0.975528,1,1,0,0,0,0,0,0
1519760,0.349647,0.990553,0.119374,-1,354,0,05-08 03:00,59.76388,21.35317,clh6aqawa0007gh0z9h6zi9bo,...,0.989074,0.396044,1,1,0,0,0,0,0,0
1519761,0.349685,0.146152,0.169276,3,50,0,05-08 03:00,59.83316,21.38489,clh6aqawa0007gh0z9h6zi9bo,...,0.128428,0.165435,1,1,0,0,0,0,0,0
1519762,0.349725,0.148930,0.173190,-1,51,0,05-08 03:00,59.89167,21.54685,clh6aqawa0007gh0z9h6zi9bo,...,0.447736,0.997261,0,0,0,0,0,0,0,0


In [78]:
# Interpolation of default rot values and normalization

train_df['rot'] = train_df['rot'].replace(128, pd.NA)

train_df['rot'] = pd.to_numeric(train_df['rot'], errors='coerce')

train_df['rot'] = train_df['rot'].interpolate(method='linear')

train_df['rot'] = train_df['rot'].ffill()
train_df['rot'] = train_df['rot'].bfill()

print("Max ROT value:", train_df['rot'].max())
print("Min ROT value:", train_df['rot'].min())
print("Number of NaN values:", train_df['rot'].isna().sum())

min_rot_value = -127
max_rot_value = 127

train_df['rot'] = (train_df['rot'] - min_rot_value) / (max_rot_value - min_rot_value)

train_df.tail()

Max ROT value: 127.0
Min ROT value: -127.0
Number of NaN values: 0


,time,cog,sog,rot,heading,navstat,etaRaw,latitude,longitude,vesselId,...,minute_sin,minute_cos,time_diff_gt_10min,time_diff_gt_20min,time_diff_gt_40min,time_diff_gt_1hour,time_diff_gt_2hours,time_diff_gt_6hours,time_diff_gt_12hours,time_diff_gt_1day
1519758,0.349568,0.900528,0.132094,0.492126,325,0,05-08 03:00,59.63337,21.43237,clh6aqawa0007gh0z9h6zi9bo,...,0.206107,0.095492,1,1,0,0,0,0,0,0
1519759,0.349607,0.900806,0.130137,0.488189,326,0,05-08 03:00,59.69588,21.34225,clh6aqawa0007gh0z9h6zi9bo,...,0.345492,0.975528,1,1,0,0,0,0,0,0
1519760,0.349647,0.990553,0.119374,0.496063,354,0,05-08 03:00,59.76388,21.35317,clh6aqawa0007gh0z9h6zi9bo,...,0.989074,0.396044,1,1,0,0,0,0,0,0
1519761,0.349685,0.146152,0.169276,0.511811,50,0,05-08 03:00,59.83316,21.38489,clh6aqawa0007gh0z9h6zi9bo,...,0.128428,0.165435,1,1,0,0,0,0,0,0
1519762,0.349725,0.148930,0.173190,0.496063,51,0,05-08 03:00,59.89167,21.54685,clh6aqawa0007gh0z9h6zi9bo,...,0.447736,0.997261,0,0,0,0,0,0,0,0


In [79]:
# Interpolation of default heading values and normalization

train_df['heading'] = train_df['heading'].replace([511, 483], pd.NA)

train_df['heading'] = pd.to_numeric(train_df['heading'], errors='coerce')

train_df['heading'] = train_df['heading'].interpolate(method='linear')

train_df['heading'] = train_df['heading'].ffill()
train_df['heading'] = train_df['heading'].bfill()

print("Max heading value:", train_df['heading'].max())
print("Min heading value:", train_df['heading'].min())
print("Number of NaN values:", train_df['heading'].isna().sum())

train_df['heading'] = train_df['heading'] / 359

train_df.tail()

Max heading value: 359.0
Min heading value: 0.0
Number of NaN values: 0


,time,cog,sog,rot,heading,navstat,etaRaw,latitude,longitude,vesselId,...,minute_sin,minute_cos,time_diff_gt_10min,time_diff_gt_20min,time_diff_gt_40min,time_diff_gt_1hour,time_diff_gt_2hours,time_diff_gt_6hours,time_diff_gt_12hours,time_diff_gt_1day
1519758,0.349568,0.900528,0.132094,0.492126,0.905292,0,05-08 03:00,59.63337,21.43237,clh6aqawa0007gh0z9h6zi9bo,...,0.206107,0.095492,1,1,0,0,0,0,0,0
1519759,0.349607,0.900806,0.130137,0.488189,0.908078,0,05-08 03:00,59.69588,21.34225,clh6aqawa0007gh0z9h6zi9bo,...,0.345492,0.975528,1,1,0,0,0,0,0,0
1519760,0.349647,0.990553,0.119374,0.496063,0.986072,0,05-08 03:00,59.76388,21.35317,clh6aqawa0007gh0z9h6zi9bo,...,0.989074,0.396044,1,1,0,0,0,0,0,0
1519761,0.349685,0.146152,0.169276,0.511811,0.139276,0,05-08 03:00,59.83316,21.38489,clh6aqawa0007gh0z9h6zi9bo,...,0.128428,0.165435,1,1,0,0,0,0,0,0
1519762,0.349725,0.148930,0.173190,0.496063,0.142061,0,05-08 03:00,59.89167,21.54685,clh6aqawa0007gh0z9h6zi9bo,...,0.447736,0.997261,0,0,0,0,0,0,0,0


In [80]:
# Navstat values


In [81]:
# Normalize lat and long?

In [82]:

print(train_df.count())
train_df.head()

time                           1519763
cog                            1519763
sog                            1519763
rot                            1519763
heading                        1519763
navstat                        1519763
etaRaw                         1519763
latitude                       1519763
longitude                      1519763
vesselId                       1519763
portId                         1519763
last_known_latitude            1519763
last_known_longitude           1519763
time_of_last_known_position    1519763
month_of_the_year              1519763
week_of_the_year               1519763
day_of_the_year                1519763
day_of_the_month               1519763
day_of_the_week                1519763
hour_of_the_day                1519763
hours_passed                   1519762
hour_sin                       1519763
hour_cos                       1519763
minute_sin                     1519763
minute_cos                     1519763
time_diff_gt_10min       

,time,cog,sog,rot,heading,navstat,etaRaw,latitude,longitude,vesselId,...,minute_sin,minute_cos,time_diff_gt_10min,time_diff_gt_20min,time_diff_gt_40min,time_diff_gt_1hour,time_diff_gt_2hours,time_diff_gt_6hours,time_diff_gt_12hours,time_diff_gt_1day
0,0.031707,0.854682,0.169276,0.519685,0.871866,0,01-14 23:30,7.57302,77.49505,61e9f38eb937134a3c4bfd8b,...,0.447736,0.002739,1,1,0,0,0,0,0,0
1,0.031757,0.852459,0.165362,0.519685,0.869081,0,01-14 23:30,7.65043,77.39404,61e9f38eb937134a3c4bfd8b,...,0.345492,0.975528,1,1,0,0,0,0,0,0
2,0.031798,0.855515,0.165362,0.523622,0.871866,0,01-14 23:30,7.71275,77.31394,61e9f38eb937134a3c4bfd8b,...,0.975528,0.345492,1,1,0,0,0,0,0,0
3,0.031838,0.853015,0.159491,0.527559,0.871866,0,01-14 23:30,7.77191,77.23585,61e9f38eb937134a3c4bfd8b,...,0.095492,0.206107,1,0,0,0,0,0,0,0
4,0.031866,0.854682,0.157534,0.519685,0.871866,0,01-14 23:30,7.81285,77.18147,61e9f38eb937134a3c4bfd8b,...,0.206107,0.904508,1,1,0,0,0,0,0,0


In [83]:
test_df.head()

,ID,vesselId,time,scaling_factor,month_of_the_year,week_of_the_year,day_of_the_year,day_of_the_month,day_of_the_week,hour_of_the_day,...,minute_sin,minute_cos,time_diff_gt_10min,time_diff_gt_20min,time_diff_gt_40min,time_diff_gt_1hour,time_diff_gt_2hours,time_diff_gt_6hours,time_diff_gt_12hours,time_diff_gt_1day
4,4,61e9f38eb937134a3c4bfd8d,0.349750,0.3,0.363636,0.346154,0.350685,0.233333,0.333333,0.000000,...,0.975528,0.654508,1,1,0,0,0,0,0,0
201,201,61e9f38eb937134a3c4bfd8d,0.349802,0.3,0.363636,0.346154,0.350685,0.233333,0.333333,0.000000,...,0.095492,0.206107,1,1,1,0,0,0,0,0
583,583,61e9f38eb937134a3c4bfd8d,0.349904,0.3,0.363636,0.346154,0.350685,0.233333,0.333333,0.043478,...,0.345492,0.024472,1,0,0,0,0,0,0,0
701,701,61e9f38eb937134a3c4bfd8d,0.349938,0.3,0.363636,0.346154,0.350685,0.233333,0.333333,0.043478,...,0.095492,0.793893,1,0,0,0,0,0,0,0
829,829,61e9f38eb937134a3c4bfd8d,0.349961,0.3,0.363636,0.346154,0.350685,0.233333,0.333333,0.086957,...,0.654508,0.975528,1,1,0,0,0,0,0,0


In [84]:
train_df.to_csv('data/processed_data/train.csv', index=False)

test_df.to_csv('data/processed_data/test.csv', index=False)
